In [ ]:
# DAY 3: MODEL BUILDING - COMPLETE VERSION
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os
import json
import pickle
import joblib
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.metrics import (mean_absolute_error, mean_squared_error, 
                           r2_score, accuracy_score, classification_report,
                           confusion_matrix)
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("DAY 3: BUILDING PREDICTION MODELS - COMPLETE")
print("="*80)
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

# ============================================================================
# 1. LOAD DATA
# ============================================================================
print("1. 📂 LOADING DATA...")
try:
    first_innings = pd.read_csv('../data/processed/first_innings_model_ready.csv')
    second_innings = pd.read_csv('../data/processed/second_innings_model_ready.csv')
    match_summary = pd.read_csv('../data/processed/match_summary.csv')
    
    print(f"   ✅ First innings: {first_innings.shape}")
    print(f"   ✅ Second innings: {second_innings.shape}")
    print(f"   ✅ Match summary: {match_summary.shape}\n")
    
except FileNotFoundError as e:
    print(f"   ❌ Error: {e}")
    print("   Please run Day 2 feature engineering first!")
    exit()

# ============================================================================
# 2. FIRST INNINGS SCORE PREDICTOR
# ============================================================================
print("2. 🎯 FIRST INNINGS SCORE PREDICTOR")
print("   " + "-"*50)

# Define features
first_features = [
    'over', 'ball', 'overs_completed', 'balls_remaining',
    'cumulative_runs', 'cumulative_wickets', 'wickets_remaining',
    'current_run_rate', 'phase', 'is_powerplay', 'is_death',
    'runs_last_5_overs', 'wickets_last_5_overs', 'rr_last_5_overs',
    'partnership_runs', 'resources_remaining'
]

# Filter available features
available_first = [f for f in first_features if f in first_innings.columns]
X_first = first_innings[available_first].copy()
y_first = first_innings['final_total']

# Handle categorical
if 'phase' in X_first.columns:
    X_first = pd.get_dummies(X_first, columns=['phase'], drop_first=True)

# Split by match
match_ids = first_innings['match_id'].unique()
train_matches, test_matches = train_test_split(match_ids, test_size=0.2, random_state=42)

train_mask = first_innings['match_id'].isin(train_matches)
test_mask = first_innings['match_id'].isin(test_matches)

X_train = X_first[train_mask]
X_test = X_first[test_mask]
y_train = y_first[train_mask]
y_test = y_first[test_mask]

print(f"   📊 Data split:")
print(f"     Training: {len(X_train):,} samples ({len(train_matches)} matches)")
print(f"     Testing: {len(X_test):,} samples ({len(test_matches)} matches)")

# Train LightGBM
print("   🤖 Training LightGBM model...")
lgb_params = {
    'objective': 'regression',
    'metric': 'mae',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': 7,
    'min_child_samples': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'n_estimators': 500,
    'random_state': 42,
    'n_jobs': -1,
    'verbosity': -1
}

lgb_model = lgb.LGBMRegressor(**lgb_params)
lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    eval_metric='mae',
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
)

# Evaluate
y_pred = lgb_model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"   📈 Performance:")
print(f"     MAE: {mae:.2f} runs")
print(f"     RMSE: {rmse:.2f} runs")
print(f"     R²: {r2:.4f}")

# Save model
os.makedirs('../models/first_innings', exist_ok=True)
joblib.dump(lgb_model, '../models/first_innings/lightgbm_model.pkl')
joblib.dump(list(X_train.columns), '../models/first_innings/feature_columns.pkl')
print("   💾 Model saved: ../models/first_innings/lightgbm_model.pkl\n")

# ============================================================================
# 3. BALL-OUTCOME PREDICTOR
# ============================================================================
print("3. 🎯 BALL-OUTCOME PREDICTOR")
print("   " + "-"*50)

# Prepare ball outcome data
all_balls = pd.concat([first_innings, second_innings], ignore_index=True)

def get_ball_outcome(row):
    if row['is_wicket'] == 1:
        return 7  # Wicket
    elif row['batsman_runs'] == 0:
        return 0  # Dot
    elif row['batsman_runs'] == 1:
        return 1  # Single
    elif row['batsman_runs'] == 2:
        return 2  # Double
    elif row['batsman_runs'] == 3:
        return 3  # Triple
    elif row['batsman_runs'] == 4:
        return 4  # Four
    elif row['batsman_runs'] == 6:
        return 6  # Six
    else:
        return 0

all_balls['ball_outcome'] = all_balls.apply(get_ball_outcome, axis=1)

# Define features
ball_features = [
    'over', 'ball', 'balls_remaining', 'wickets_remaining',
    'cumulative_runs', 'cumulative_wickets', 'current_run_rate',
    'runs_needed', 'required_run_rate', 'runs_ahead',
    'phase', 'is_powerplay', 'is_death',
    'runs_last_5_overs', 'wickets_last_5_overs',
    'partnership_runs', 'balls_since_last_wicket',
    'resources_remaining', 'pressure_index', 'rr_ratio'
]

available_ball = [f for f in ball_features if f in all_balls.columns]
X_ball = all_balls[available_ball].copy()
y_ball = all_balls['ball_outcome']

if 'phase' in X_ball.columns:
    X_ball = pd.get_dummies(X_ball, columns=['phase'], drop_first=True)

# Split
ball_match_ids = all_balls['match_id'].unique()
train_ball_matches, test_ball_matches = train_test_split(
    ball_match_ids, test_size=0.2, random_state=42
)

train_ball_mask = all_balls['match_id'].isin(train_ball_matches)
test_ball_mask = all_balls['match_id'].isin(test_ball_matches)

X_ball_train = X_ball[train_ball_mask]
X_ball_test = X_ball[test_ball_mask]
y_ball_train = y_ball[train_ball_mask]
y_ball_test = y_ball[test_ball_mask]

print(f"   📊 Data split:")
print(f"     Training: {len(X_ball_train):,} balls")
print(f"     Testing: {len(X_ball_test):,} balls")

# Train Random Forest
print("   🤖 Training Random Forest classifier...")
rf_params = {
    'n_estimators': 200,
    'max_depth': 10,
    'min_samples_split': 20,
    'min_samples_leaf': 10,
    'max_features': 'sqrt',
    'bootstrap': True,
    'random_state': 42,
    'n_jobs': -1,
    'class_weight': 'balanced'
}

rf_model = RandomForestClassifier(**rf_params)
rf_model.fit(X_ball_train, y_ball_train)

# Evaluate
y_ball_pred = rf_model.predict(X_ball_test)
accuracy = accuracy_score(y_ball_test, y_ball_pred)

outcome_names = ['Dot', 'Single', 'Double', 'Triple', 'Four', 'Six', 'Wicket']
print(f"   📈 Performance:")
print(f"     Accuracy: {accuracy:.4f}")
print("\n   📋 Classification Report:")
print(classification_report(y_ball_test, y_ball_pred, 
                          target_names=outcome_names, digits=3))

# Save model
os.makedirs('../models/ball_outcome', exist_ok=True)
joblib.dump(rf_model, '../models/ball_outcome/random_forest_model.pkl')
joblib.dump(list(X_ball_train.columns), '../models/ball_outcome/feature_columns.pkl')
print("   💾 Model saved: ../models/ball_outcome/random_forest_model.pkl\n")

# ============================================================================
# 4. WIN PROBABILITY ENGINE
# ============================================================================
print("4. 🎯 WIN PROBABILITY ENGINE")
print("   " + "-"*50)

class WinProbabilityEngine:
    def __init__(self, first_innings_model, ball_outcome_model):
        self.first_innings_model = first_innings_model
        self.ball_outcome_model = ball_outcome_model
        self.outcome_names = outcome_names
        
    def predict_win_probability(self, match_state, n_simulations=5000):
        """Calculate win probability for given match state"""
        
        # Extract state variables
        inning = match_state.get('inning', 1)
        current_runs = match_state.get('current_runs', 0)
        current_wickets = match_state.get('current_wickets', 0)
        balls_remaining = match_state.get('balls_remaining', 120)
        target = match_state.get('target', None)
        
        if inning == 1:
            # First innings - predict final score
            return self._predict_first_innings(match_state, n_simulations)
        else:
            # Second innings - simulate chase
            return self._simulate_chase(match_state, n_simulations)
    
    def _predict_first_innings(self, match_state, n_simulations):
        """Predict final score for first innings"""
        # Create features for first innings model
        features = self._create_features(match_state, is_first_innings=True)
        
        # Predict using LightGBM model
        predicted_score = self.first_innings_model.predict(features)[0]
        
        # Add uncertainty (normal distribution around prediction)
        scores = np.random.normal(predicted_score, 15, n_simulations)
        scores = np.maximum(scores, match_state['current_runs'])
        
        return {
            'predicted_score': float(predicted_score),
            'confidence_interval': (float(np.percentile(scores, 10)), 
                                   float(np.percentile(scores, 90))),
            'score_distribution': scores.tolist()
        }
    
    def _simulate_chase(self, match_state, n_simulations):
        """Simulate chase for second innings"""
        current_runs = match_state['current_runs']
        current_wickets = match_state['current_wickets']
        balls_remaining = match_state['balls_remaining']
        target = match_state['target']
        
        wins = 0
        final_scores = []
        
        for _ in range(n_simulations):
            runs = current_runs
            wickets = current_wickets
            balls = balls_remaining
            
            # Simulate ball by ball
            while balls > 0 and wickets < 10:
                # Create state for this ball
                ball_state = match_state.copy()
                ball_state['current_runs'] = runs
                ball_state['current_wickets'] = wickets
                ball_state['balls_remaining'] = balls
                
                # Predict ball outcome
                features = self._create_features(ball_state, is_first_innings=False)
                
                try:
                    # Use Random Forest probabilities
                    if hasattr(self.ball_outcome_model, 'predict_proba'):
                        probs = self.ball_outcome_model.predict_proba(features)[0]
                        outcome = np.random.choice(len(probs), p=probs)
                    else:
                        # Fallback to simple distribution
                        outcome = self._get_simple_outcome(ball_state)
                except:
                    outcome = self._get_simple_outcome(ball_state)
                
                # Update based on outcome
                if outcome == 7:  # Wicket
                    wickets += 1
                elif outcome in [1, 2, 3, 4, 6]:
                    runs += outcome
                # Dot ball (0) adds nothing
                
                balls -= 1
                
                # Early exit if target achieved
                if target is not None and runs >= target:
                    break
            
            final_scores.append(runs)
            if target is not None and runs >= target:
                wins += 1
        
        win_prob = wins / n_simulations if target is not None else None
        
        return {
            'win_probability': float(win_prob) if win_prob is not None else None,
            'average_final_score': float(np.mean(final_scores)),
            'wins': int(wins),
            'simulations': int(n_simulations)
        }
    
    def _create_features(self, match_state, is_first_innings=False):
        """Create feature vector from match state"""
        # Basic features
        features = {
            'over': match_state.get('over', 0),
            'ball': match_state.get('ball', 1),
            'overs_completed': match_state.get('overs_completed', 
                                             match_state.get('over', 0) + 
                                             match_state.get('ball', 1)/10),
            'balls_remaining': match_state.get('balls_remaining', 120),
            'cumulative_runs': match_state.get('current_runs', 0),
            'cumulative_wickets': match_state.get('current_wickets', 0),
            'wickets_remaining': 10 - match_state.get('current_wickets', 0),
        }
        
        # Calculate run rate
        if features['overs_completed'] > 0:
            features['current_run_rate'] = (features['cumulative_runs'] / 
                                          features['overs_completed'])
        else:
            features['current_run_rate'] = 0
        
        # Phase features
        features['is_powerplay'] = 1 if features['over'] < 6 else 0
        features['is_death'] = 1 if features['over'] >= 15 else 0
        
        # For second innings
        if not is_first_innings:
            features['runs_needed'] = match_state.get('runs_needed', 0)
            features['required_run_rate'] = match_state.get('required_run_rate', 0)
            features['runs_ahead'] = match_state.get('runs_ahead', 0)
            features['target'] = match_state.get('target', 0)
        
        # Convert to DataFrame with one row
        df = pd.DataFrame([features])
        
        # Add dummy columns for phase if needed
        if 'phase_middle' in self.ball_outcome_model.feature_names_in_:
            df['phase_middle'] = 1 if 6 <= features['over'] < 15 else 0
            df['phase_powerplay'] = features['is_powerplay']
            df['phase_death'] = features['is_death']
        
        # Ensure all columns exist
        for col in self.ball_outcome_model.feature_names_in_:
            if col not in df.columns:
                df[col] = 0
        
        return df[self.ball_outcome_model.feature_names_in_]
    
    def _get_simple_outcome(self, match_state):
        """Simple fallback outcome distribution"""
        over = match_state.get('over', 0)
        
        if over < 6:  # Powerplay
            probs = [0.30, 0.25, 0.15, 0.02, 0.20, 0.05, 0.03]
        elif over < 15:  # Middle
            probs = [0.25, 0.30, 0.15, 0.01, 0.20, 0.06, 0.03]
        else:  # Death
            probs = [0.20, 0.25, 0.10, 0.01, 0.25, 0.15, 0.04]
        
        return np.random.choice(range(7), p=probs)

# Create and test the engine
print("   🤖 Creating Win Probability Engine...")
engine = WinProbabilityEngine(lgb_model, rf_model)

# Test Case 1: First innings
print("\n   🧪 Test Case 1 - First Innings:")
state_1 = {
    'inning': 1,
    'over': 10,
    'ball': 3,
    'current_runs': 85,
    'current_wickets': 3,
    'balls_remaining': 57
}
result_1 = engine.predict_win_probability(state_1, n_simulations=1000)
print(f"     State: {state_1['current_runs']}/{state_1['current_wickets']} after {state_1['over']}.{state_1['ball']}")
print(f"     Predicted final score: {result_1['predicted_score']:.1f}")
print(f"     80% CI: [{result_1['confidence_interval'][0]:.0f}, {result_1['confidence_interval'][1]:.0f}]")

# Test Case 2: Second innings
print("\n   🧪 Test Case 2 - Second Innings:")
state_2 = {
    'inning': 2,
    'over': 15,
    'ball': 3,
    'current_runs': 120,
    'current_wickets': 4,
    'balls_remaining': 27,
    'target': 162,
    'runs_needed': 42,
    'required_run_rate': 9.33
}
result_2 = engine.predict_win_probability(state_2, n_simulations=2000)
print(f"     State: {state_2['current_runs']}/{state_2['current_wickets']}")
print(f"     Target: {state_2['target']}, Need: {state_2['runs_needed']} in {state_2['balls_remaining']} balls")
print(f"     Win Probability: {result_2['win_probability']:.2%}")
print(f"     Average final score: {result_2['average_final_score']:.1f}")

# Save the engine
joblib.dump(engine, '../models/win_probability_engine.pkl')
print("\n   💾 Engine saved: ../models/win_probability_engine.pkl")

# ============================================================================
# 5. VALIDATION ON HISTORICAL MATCHES
# ============================================================================
print("\n5. 📊 VALIDATION ON HISTORICAL MATCHES")
print("   " + "-"*50)

# Test on 3 random matches
test_matches_list = list(test_matches)[:3]

for match_id in test_matches_list:
    match_data = second_innings[second_innings['match_id'] == match_id]
    if len(match_data) < 10:
        continue
    
    match_info = match_summary[match_summary['match_id'] == match_id]
    if len(match_info) == 0:
        continue
    
    print(f"\n   🏏 Match {match_id}:")
    print(f"     {match_info['team1'].iloc[0]} vs {match_info['team2'].iloc[0]}")
    print(f"     Score: {match_info['team1_score'].iloc[0]}/{match_info['team2_score'].iloc[0]}")
    
    # Get actual result
    batting_team = match_data['batting_team'].iloc[0]
    winner = match_info['winner'].iloc[0]
    actual_win = 1 if batting_team == winner else 0
    print(f"     Result: {'Batting team WON' if actual_win == 1 else 'Batting team LOST'}")
    
    # Test at key points
    test_points = [0, 30, 60, 90, 120]
    print("\n     Over-by-over Win Probability:")
    print("     " + "="*60)
    print("     Over | Score | Wkts | Need | Win Prob")
    print("     " + "-"*60)
    
    for point in test_points:
        if point < len(match_data):
            ball_data = match_data.iloc[point]
            
            state = {
                'inning': 2,
                'over': ball_data['over'],
                'ball': ball_data['ball'],
                'current_runs': ball_data['cumulative_runs'],
                'current_wickets': ball_data['cumulative_wickets'],
                'balls_remaining': ball_data['balls_remaining'],
                'target': ball_data['target'],
                'runs_needed': ball_data['runs_needed'],
                'required_run_rate': ball_data['required_run_rate']
            }
            
            result = engine.predict_win_probability(state, n_simulations=1000)
            win_prob = result['win_probability']
            
            print(f"     {ball_data['over']:2}.{ball_data['ball']:<2} | "
                  f"{ball_data['cumulative_runs']:3d}/{ball_data['cumulative_wickets']:<2} | "
                  f"{ball_data['runs_needed']:4d} | {win_prob:7.2%}")

print("\n" + "="*80)
print("DAY 3 COMPLETED SUCCESSFULLY! 🎉")
print("="*80)
print("\n📋 MODELS CREATED:")
print("   1. First Innings Score Predictor (LightGBM)")
print("   2. Ball-Outcome Predictor (Random Forest)")
print("   3. Win Probability Engine (Monte Carlo Simulation)")
print("\n📁 FILES SAVED:")
print("   • models/first_innings/lightgbm_model.pkl")
print("   • models/ball_outcome/random_forest_model.pkl")
print("   • models/win_probability_engine.pkl")
print(f"\n⏰ End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

_IncompleteInputError: incomplete input (3110100197.py, line 765)